In [2]:
!pip install sentence-transformers
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 64.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [12]:
import faiss, pandas as pd, numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

In [6]:
model = SentenceTransformer("all-MiniLM-L12-v2") # add more. All of the BERT family uses only the encoder.

In [7]:
# Load the dataset
text_path = "ages.txt" # back in the pycharm environment it will have to change to data/ages.txt

sentences = []
with open(text_path, "r") as f:
    for line in f:
        sentences.append(line.strip())

print("Number of sentences:", len(sentences))
print(sentences[:10])

Number of sentences: 100
['My age is 91', 'My age is 94', 'My age is 27', 'My age is 85', 'My age is 67', 'My age is 84', 'My age is 71', 'My age is 85', 'My age is 61', 'My age is 78']


In [11]:
"""embeddings = []
BATCH = 128
for i in tqdm(range(0, len(sentences), BATCH)):
  batch = sentences[i:i+BATCH]
  emb = model.encode(batch)
  embeddings.append(emb)

embeddings = np.vstack(embeddings)

index = faiss.IndexFlatL2(model.get_sentence_embedding_dimension())
index.add(embeddings)"""

100%|██████████| 1/1 [00:00<00:00,  1.32it/s]


In [19]:
def failproof(query_age):
  return min(sentences, key=lambda x: abs(int(x.split()[-1]) - query_age))

In [20]:
def simplest_query(age:int, sentences):
  query_age = str(age)
  emb_query = model.encode(query_age, convert_to_tensor=True)
  emb_sentences = model.encode(sentences, convert_to_tensor=True)
  #distances = faiss.search(emb_query, emb_sentences)
  similarities = util.cos_sim(emb_query, emb_sentences)[0]

  best_idx = int(similarities.argmax())
  return sentences[best_idx]

In [21]:
def compare(age, sentences):
  return simplest_query(age, sentences), failproof(age)

In [24]:
transformer, math = compare(70, sentences)
print("transformer: ", transformer)
print("math: ", math)

transformer:  My age is 78
math:  My age is 71


In [18]:
print(simplest_query(70, sentences))

My age is 78
